---
title: "1 - Baseline Multi-Year Tesla 10-K RAG"
jupyter: python3
execute:
  eval: false
  echo: true
  warning: false
  message: false
---

# Baseline Multi-Year Tesla 10-K RAG

This notebook builds a simple Retrieval-Augmented Generation pipeline over Tesla 10-K filings from 2015 through 2025.

The baseline has one purpose: make the evidence path visible. It downloads or reuses cached SEC filings, chunks the filings with year metadata, embeds the chunks with a Hugging Face sentence-transformer model, retrieves relevant chunks with FAISS, and generates an answer that cites filing years and chunk IDs.

## Setup

In [1]:
%pip install -q beautifulsoup4 requests pandas faiss-cpu sentence-transformers langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface langchain-openai

Note: you may need to restart the kernel to use updated packages.


D:\Repositories\AD698-generative-ai-for-BA\.venv\Scripts\python.exe: No module named pip


## Inline SEC 10-K Helper Code

The next cell is intentionally kept inside the notebook instead of being imported from a separate module. It shows the full mechanics of the RAG data layer:

- the multi-year Tesla 10-K URL list
- SEC download and local caching
- HTML-to-text cleanup
- LangChain `Document` creation with year metadata
- recursive chunking with stable `chunk_id` values
- retrieval diagnostics for keyword coverage and year coverage
- context formatting for grounded answer generation

Keeping this code visible makes the notebooks more demonstrative: the retrieval and evaluation results can be traced back to the exact preprocessing choices.

In [2]:
from pathlib import Path
import os
import shutil
import sys
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

if (Path.cwd() / "help-code" / "Rag").exists():
    RAG_DIR = Path.cwd() / "help-code" / "Rag"
else:
    RAG_DIR = Path.cwd()

IN_COLAB = "google.colab" in sys.modules

# Colab Drive is optional. Uncomment these lines in Colab when you want
# SEC filings and output CSVs mirrored to Google Drive.
# from google.colab import drive
# drive.mount("/content/drive")

DRIVE_RAG_DIR = Path("/content/drive/MyDrive/Colab Notebooks/RAG Models")


def get_secret(name: str, aliases: list[str] | None = None) -> str | None:
    """Return a secret from environment variables or Colab userdata."""
    names = [name, *(aliases or [])]
    for candidate in names:
        value = os.environ.get(candidate)
        if value:
            return value
    try:
        from google.colab import userdata
        for candidate in names:
            value = userdata.get(candidate)
            if value:
                return value
    except Exception:
        pass
    return None


def load_secret_to_env(env_name: str, aliases: list[str] | None = None) -> None:
    value = get_secret(env_name, aliases=aliases)
    if value and env_name not in os.environ:
        os.environ[env_name] = value


load_secret_to_env("OPENAI_API_KEY", aliases=["OPENAI_KEY"])
load_secret_to_env("COHERE_API_KEY", aliases=["COHERE_KEY"])
load_secret_to_env("HF_TOKEN", aliases=["HUGGINGFACE_API_KEY", "HF_KEY"])


def default_sec_cache_dir() -> Path:
    """Choose a cache path that works in Colab and locally."""
    if DRIVE_RAG_DIR.exists():
        return DRIVE_RAG_DIR / "sec-cache" / "sec-edgar-filings"
    repo_root = find_repo_root()
    if (repo_root / "help-code").exists():
        return repo_root / "help-code" / "secfile" / "sec-edgar-filings"
    return Path.cwd() / "sec-cache" / "sec-edgar-filings"


def get_output_dirs() -> tuple[Path, Path | None]:
    local_output_dir = RAG_DIR / "outputs"
    drive_output_dir = DRIVE_RAG_DIR / "outputs" if DRIVE_RAG_DIR.exists() else None
    local_output_dir.mkdir(parents=True, exist_ok=True)
    if drive_output_dir:
        drive_output_dir.mkdir(parents=True, exist_ok=True)
    return local_output_dir, drive_output_dir


def mirror_outputs_to_drive(local_output_dir: Path) -> None:
    _, drive_output_dir = get_output_dirs()
    if not drive_output_dir:
        return
    for path in local_output_dir.glob("*.csv"):
        shutil.copy2(path, drive_output_dir / path.name)

from dataclasses import dataclass
from typing import Iterable
from urllib.parse import urlparse
import re
import time
import numpy as np
import requests
from bs4 import BeautifulSoup
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

SEC_USER_AGENT = "AD698-RAG-course/1.0 contact@example.com"

TESLA_10K_URLS = [
    "https://www.sec.gov/Archives/edgar/data/1318605/000156459016013195/tsla-10k_20151231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000156459017003118/tsla-10k_20161231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000156459018002956/tsla-10k_20171231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000156459019003165/tsla-10k_20181231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000156459020004475/tsla-10k_20191231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000156459021004599/tsla-10k_20201231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000095017022000796/tsla-20211231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000095017023001409/tsla-20221231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000162828024002390/tsla-20231231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000162828025003063/tsla-20241231.htm",
    "https://www.sec.gov/Archives/edgar/data/1318605/000162828026003952/tsla-20251231.htm",
]

TESLA_EVAL_QUESTIONS = [
    {
        "qid": "q1_battery_technology",
        "user_input": "How did Tesla describe battery technology, battery costs, or battery supply across the filings?",
        "reference": "A strong answer compares multiple years and cites evidence about battery technology, battery cost, supply constraints, production scale, and energy storage or vehicle battery strategy.",
        "reference_keywords": ["battery", "cost", "supply", "technology", "production"],
    },
    {
        "qid": "q2_manufacturing_capacity",
        "user_input": "How did Tesla's manufacturing capacity and production ramp risks evolve from 2015 through 2025?",
        "reference": "A strong answer discusses production ramp, manufacturing capacity, factories, scaling risk, delivery volume, and operational execution across several filing years.",
        "reference_keywords": ["manufacturing", "production", "capacity", "factory", "ramp"],
    },
    {
        "qid": "q3_autopilot_self_driving",
        "user_input": "What do the filings say about Autopilot, self-driving, artificial intelligence, or autonomous vehicle technology?",
        "reference": "A strong answer cites filing evidence about Autopilot, Full Self-Driving, autonomous driving, AI systems, safety, regulatory risk, and product development.",
        "reference_keywords": ["autopilot", "self-driving", "autonomous", "artificial intelligence", "regulatory"],
    },
    {
        "qid": "q4_regulatory_risk",
        "user_input": "What regulatory risks appear repeatedly in Tesla's multi-year 10-K filings?",
        "reference": "A strong answer identifies recurring regulatory themes such as vehicle safety, emissions, energy, consumer protection, data, labor, international operations, and securities or compliance risk.",
        "reference_keywords": ["regulatory", "compliance", "safety", "emissions", "international"],
    },
    {
        "qid": "q5_revenue_business_model",
        "user_input": "How did Tesla's business model and revenue sources change across the filing years?",
        "reference": "A strong answer compares automotive revenue with energy generation and storage, services, leasing, regulatory credits, growth in deliveries, and changes in operating scale.",
        "reference_keywords": ["revenue", "automotive", "energy", "services", "leasing"],
    },
    {
        "qid": "q6_competition",
        "user_input": "How does Tesla describe competitive pressure in electric vehicles, energy storage, and related markets?",
        "reference": "A strong answer cites competition from incumbent automakers, EV entrants, battery suppliers, energy storage providers, technology companies, and pricing or innovation pressure.",
        "reference_keywords": ["competition", "electric vehicles", "energy storage", "pricing", "innovation"],
    },
]

@dataclass(frozen=True)
class FilingRecord:
    ticker: str
    year: int
    url: str
    local_path: Path


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in [start, *start.parents]:
        if (candidate / "help-code").exists() and (candidate / "M5").exists():
            return candidate
    return Path.cwd()


def filing_year_from_url(url: str) -> int:
    match = re.search(r"20\d{2}|19\d{2}", url)
    if not match:
        raise ValueError(f"Could not infer year from URL: {url}")
    return int(match.group(0))


def cache_path_for_url(url: str, cache_dir: Path) -> Path:
    parsed = urlparse(url)
    filename = Path(parsed.path).name
    year = filing_year_from_url(url)
    return cache_dir / "TSLA" / "10-K" / str(year) / filename


def tesla_filing_records(cache_dir: Path | None = None) -> list[FilingRecord]:
    root = find_repo_root()
    cache_dir = default_sec_cache_dir() if cache_dir is None else cache_dir
    return [
        FilingRecord(
            ticker="TSLA",
            year=filing_year_from_url(url),
            url=url,
            local_path=cache_path_for_url(url, cache_dir),
        )
        for url in TESLA_10K_URLS
    ]


def download_filing(record: FilingRecord, user_agent: str = SEC_USER_AGENT, sleep_seconds: float = 0.2) -> Path:
    record.local_path.parent.mkdir(parents=True, exist_ok=True)
    if record.local_path.exists() and record.local_path.stat().st_size > 0:
        return record.local_path
    response = requests.get(record.url, headers={"User-Agent": user_agent}, timeout=60)
    response.raise_for_status()
    record.local_path.write_text(response.text, encoding="utf-8", errors="ignore")
    time.sleep(sleep_seconds)
    return record.local_path


def ensure_tesla_filings(download: bool = True, user_agent: str = SEC_USER_AGENT) -> list[FilingRecord]:
    records = tesla_filing_records()
    if download:
        for record in records:
            download_filing(record, user_agent=user_agent)
    return records


def html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "ix:header", "ix:hidden"]):
        tag.decompose()
    text = soup.get_text(" ")
    return re.sub(r"\s+", " ", text).strip()


def load_filing_documents(records: Iterable[FilingRecord], max_chars_per_filing: int | None = None) -> list[Document]:
    docs = []
    for record in records:
        if not record.local_path.exists():
            raise FileNotFoundError(f"Missing cached filing for {record.year}: {record.local_path}")
        html = record.local_path.read_text(encoding="utf-8", errors="ignore")
        text = html_to_text(html)
        if max_chars_per_filing:
            text = text[:max_chars_per_filing]
        docs.append(
            Document(
                page_content=text,
                metadata={
                    "ticker": record.ticker,
                    "filing_year": record.year,
                    "filing_type": "10-K",
                    "source_url": record.url,
                    "source_file": str(record.local_path),
                },
            )
        )
    return docs


def chunk_filings(docs: list[Document], chunk_size: int = 1200, chunk_overlap: int = 180) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["Item 1A.", "Item 7.", "Item 8.", "\n\n", ". ", " "],
    )
    chunks = splitter.split_documents(docs)
    for i, doc in enumerate(chunks):
        year = doc.metadata.get("filing_year", "unknown")
        ticker = doc.metadata.get("ticker", "TSLA")
        doc.metadata["chunk_id"] = f"{ticker}-{year}-{i:05d}"
    return chunks


def chunk_audit(chunks: list[Document]) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "chunk_id": doc.metadata.get("chunk_id"),
            "ticker": doc.metadata.get("ticker"),
            "filing_year": doc.metadata.get("filing_year"),
            "n_chars": len(doc.page_content),
            "preview": doc.page_content[:180],
        }
        for doc in chunks
    )


def format_context(docs: list[Document], max_chars_per_doc: int = 1600) -> str:
    return "\n\n".join(
        f"[{doc.metadata.get('ticker')} {doc.metadata.get('filing_year')} {doc.metadata.get('chunk_id')}]\n"
        f"{doc.page_content[:max_chars_per_doc]}"
        for doc in docs
    )


def keyword_hit(text: str, keyword: str) -> bool:
    return keyword.lower() in re.sub(r"\s+", " ", text.lower())


def retrieval_diagnostics(retrieved_docs: list[Document], reference_keywords: list[str]) -> dict:
    combined = " ".join(doc.page_content for doc in retrieved_docs)
    keyword_hits = {kw: keyword_hit(combined, kw) for kw in reference_keywords}
    rank_weighted_hits = []
    for rank, doc in enumerate(retrieved_docs, start=1):
        hits = sum(keyword_hit(doc.page_content, kw) for kw in reference_keywords)
        rank_weighted_hits.append(hits / rank)
    years = [doc.metadata.get("filing_year") for doc in retrieved_docs]
    return {
        "retrieved_years": years,
        "unique_years_retrieved": len(set(years)),
        "keyword_coverage": float(np.mean(list(keyword_hits.values()))) if keyword_hits else np.nan,
        "rank_weighted_keyword_score": float(np.sum(rank_weighted_hits)),
        "keyword_hits": keyword_hits,
    }


def extractive_fallback_answer(question: str, retrieved_docs: list[Document], max_chars_per_doc: int = 500) -> str:
    parts = [f"Question: {question}", "Retrieved Tesla filing evidence:"]
    for doc in retrieved_docs[:4]:
        parts.append(
            f"[TSLA {doc.metadata.get('filing_year')} {doc.metadata.get('chunk_id')}] "
            f"{doc.page_content[:max_chars_per_doc]}"
        )
    return "\n\n".join(parts)

C:\Users\nakulpadalkar\AppData\Local\Temp\ipykernel_53708\1121230134.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Download or Reuse Cached Tesla Filings

The SEC requests use a descriptive user agent. Files are cached under `help-code/secfile/sec-edgar-filings/TSLA/10-K/{year}/`.

In [3]:
records = ensure_tesla_filings(download=True)
pd.DataFrame(
    {
        "year": record.year,
        "url": record.url,
        "local_path": str(record.local_path),
        "cached": record.local_path.exists(),
    }
    for record in records
)

,year,url,local_path,cached
0,2015,https://www.sec.gov/Archives/edgar/data/131860...,D:\Repositories\AD698-generative-ai-for-BA\hel...,True
1,2016,https://www.sec.gov/Archives/edgar/data/131860...,D:\Repositories\AD698-generative-ai-for-BA\hel...,True
2,2017,https://www.sec.gov/Archives/edgar/data/131860...,D:\Repositories\AD698-generative-ai-for-BA\hel...,True
3,1900,https://www.sec.gov/Archives/edgar/data/131860...,D:\Repositories\AD698-generative-ai-for-BA\hel...,True
4,2000,https://www.sec.gov/Archives/edgar/data/131860...,D:\Repositories\AD698-generative-ai-for-BA\hel...,True
5,2020,https://www.sec.gov/Archives/edgar/data/131860...,D:\Repositories\AD698-generative-ai-for-BA\hel...,True
6,2000,https://www.sec.gov/Archives/edgar/data/131860...,D:\Repositories\AD698-generative-ai-for-BA\hel...,True
7,2022,https://www.sec.gov/Archives/edgar/data/131860...,D:\Repositories\AD698-generative-ai-for-BA\hel...,True
8,2023,https://www.sec.gov/Archives/edgar/data/131860...,D:\Repositories\AD698-generative-ai-for-BA\hel...,True
9,2024,https://www.sec.gov/Archives/edgar/data/131860...,D:\Repositories\AD698-generative-ai-for-BA\hel...,True


## Load and Chunk the Filings

In [4]:
docs = load_filing_documents(records)
chunks = chunk_filings(docs, chunk_size=1200, chunk_overlap=180)

audit_df = chunk_audit(chunks)
audit_df.groupby("filing_year").agg(
    chunks=("chunk_id", "count"),
    avg_chars=("n_chars", "mean"),
    min_chars=("n_chars", "min"),
    max_chars=("n_chars", "max"),
)

,chunks,avg_chars,min_chars,max_chars
filing_year,,,,
1900,622,1038.286174,15,1200
2000,1015,1041.668966,15,1200
2015,353,1030.917847,22,1200
2016,564,1044.049645,15,1200
2017,553,1040.108499,15,1200
2020,526,1046.294677,139,1200
2022,420,1038.642857,88,1200
2023,408,1038.752451,15,1200
2024,391,1043.583120,15,1200


## Build the Baseline Vector Index

In [5]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)

vector_store = FAISS.from_documents(chunks, embedding_model)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 6})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Retrieval Audit

In [6]:
question = TESLA_EVAL_QUESTIONS[0]["user_input"]
retrieved_docs = retriever.invoke(question)

pd.DataFrame(
    {
        "rank": rank,
        "year": doc.metadata["filing_year"],
        "chunk_id": doc.metadata["chunk_id"],
        "preview": doc.page_content[:260],
    }
    for rank, doc in enumerate(retrieved_docs, start=1)
)

,rank,year,chunk_id,preview
0,1,2015,TSLA-2015-00039,". Of this, we expect to build 35 GWh of cell p..."
1,2,2015,TSLA-2015-00018,. The Tesla Energy product portfolio will incl...
2,3,2017,TSLA-2017-00981,". Additionally, we previously announced a pate..."
3,4,2024,TSLA-2024-04620,". As these product lines grow, we will have to..."
4,5,2015,TSLA-2015-00035,. We introduced this program in North America ...
5,6,2015,TSLA-2015-00014,". In addition to developing our own vehicles, ..."


In [7]:
retrieval_diagnostics(
    retrieved_docs,
    reference_keywords=TESLA_EVAL_QUESTIONS[0]["reference_keywords"],
)

{'retrieved_years': [2015, 2015, 2017, 2024, 2015, 2015],
 'unique_years_retrieved': 3,
 'keyword_coverage': 1.0,
 'rank_weighted_keyword_score': 7.033333333333333,
 'keyword_hits': {'battery': True,
  'cost': True,
  'supply': True,
  'technology': True,
  'production': True}}

## Generate a Grounded Answer

The generator cell uses OpenAI only if `OPENAI_API_KEY` is set. Without a key, the notebook still returns an extractive evidence summary so the retrieval path can be inspected.

In [8]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        (
            "Answer using only the retrieved Tesla 10-K context. "
            "Cite filing years and chunk IDs. "
            "If the retrieved context is insufficient, say so."
        ),
    ),
    ("user", "Question:\n{question}\n\nContext:\n{context}"),
])

if "OPENAI_API_KEY" in os.environ:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    answer_chain = prompt | llm | StrOutputParser()
    answer = answer_chain.invoke(
        {"question": question, "context": format_context(retrieved_docs)}
    )
else:
    answer = extractive_fallback_answer(question, retrieved_docs)

print(answer)

Question: How did Tesla describe battery technology, battery costs, or battery supply across the filings?

Retrieved Tesla filing evidence:

[TSLA 2015 TSLA-2015-00039] . Of this, we expect to build 35 GWh of cell production capacity at the Gigafactory and purchase 15 GWh of cells from other manufacturers. We believe that the Gigafactory will allow us to achieve a significant reduction in the cost of our battery packs once we are in volume production with Model 3. The total capital expenditures associated with the Gigafactory through 2020 are expected to be $4 to $5 billion, of which approximately $2 billion is expected to come from Tesla. Panasonic has agreed 

[TSLA 2015 TSLA-2015-00018] . The Tesla Energy product portfolio will include energy storage products with a wide range of applications, from use in single homes to use in larger utility-scale projects. Tesla Powerwall is a rechargeable lithium-ion battery designed to store energy at a residential and small commercial level for

## Run the Baseline Over the Evaluation Questions

In [9]:
rows = []
for item in TESLA_EVAL_QUESTIONS:
    docs_for_q = retriever.invoke(item["user_input"])
    if "OPENAI_API_KEY" in os.environ:
        response = answer_chain.invoke(
            {"question": item["user_input"], "context": format_context(docs_for_q)}
        )
        response_mode = "generated"
    else:
        response = extractive_fallback_answer(item["user_input"], docs_for_q)
        response_mode = "extractive_fallback"

    diag = retrieval_diagnostics(docs_for_q, item["reference_keywords"])
    rows.append(
        {
            "qid": item["qid"],
            "question": item["user_input"],
            "response": response,
            "response_mode": response_mode,
            "retrieved_years": diag["retrieved_years"],
            "keyword_coverage": diag["keyword_coverage"],
            "rank_weighted_keyword_score": diag["rank_weighted_keyword_score"],
        }
    )

baseline_results = pd.DataFrame(rows)
baseline_results

,qid,question,response,response_mode,retrieved_years,keyword_coverage,rank_weighted_keyword_score
0,q1_battery_technology,"How did Tesla describe battery technology, bat...",Question: How did Tesla describe battery techn...,extractive_fallback,"[2015, 2015, 2017, 2024, 2015, 2015]",1.0,7.033333
1,q2_manufacturing_capacity,How did Tesla's manufacturing capacity and pro...,Question: How did Tesla's manufacturing capaci...,extractive_fallback,"[2015, 1900, 2022, 2015, 2016, 2024]",1.0,7.483333
2,q3_autopilot_self_driving,"What do the filings say about Autopilot, self-...",Question: What do the filings say about Autopi...,extractive_fallback,"[2017, 1900, 2024, 2023, 1900, 2000]",0.8,5.100000
3,q4_regulatory_risk,What regulatory risks appear repeatedly in Tes...,Question: What regulatory risks appear repeate...,extractive_fallback,"[2020, 2000, 2023, 2022, 1900, 2000]",0.8,4.350000
4,q5_revenue_business_model,How did Tesla's business model and revenue sou...,Question: How did Tesla's business model and r...,extractive_fallback,"[2015, 2015, 2015, 2016, 2000, 2017]",1.0,7.166667
5,q6_competition,How does Tesla describe competitive pressure i...,Question: How does Tesla describe competitive ...,extractive_fallback,"[2017, 2020, 2000, 1900, 2000, 2023]",0.6,4.116667


## Save Baseline Artifacts

In [10]:
output_dir, drive_output_dir = get_output_dirs()

baseline_results.to_csv(output_dir / "tesla_baseline_rag_results.csv", index=False)
audit_df.to_csv(output_dir / "tesla_chunk_audit.csv", index=False)

mirror_outputs_to_drive(output_dir)

output_dir, drive_output_dir

(WindowsPath('D:/Repositories/AD698-generative-ai-for-BA/help-code/Rag/outputs'),
 None)

## Reading the Baseline

The baseline is intentionally simple. It gives a first view of:

- which filing years are retrieved for each question
- whether the retrieved chunks contain expected keywords
- whether answer generation is faithful to the retrieved context
- which questions need reranking, decomposition, or evaluation